In [1]:
# import necessary libraries
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
import nltk
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from bertopic import BERTopic
from bertopic.representation import MaximalMarginalRelevance
import re
import html
import random
import torch
from transformers import set_seed
from hdbscan import HDBSCAN
import umap
import seaborn as sns
pd.set_option("display.max_colwidth", 200)
sns.set(style="whitegrid")
nltk.download("wordnet")
nltk.download("omw-1.4")
from sklearn.metrics.pairwise import cosine_similarity
from IPython.display import display, Markdown, clear_output
import ipywidgets as widgets

# ensure reproducibility
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
set_seed(SEED)

[nltk_data] Downloading package wordnet to
[nltk_data]     /Users/rachelrieille/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     /Users/rachelrieille/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


## Task 2.1

In [ ]:
# Load datasets
reviews = pd.read_csv("ATML2025_book_reviews_train.csv")         
book_info = pd.read_csv("ATML2025_books.csv")   
merged = pd.merge(reviews, book_info, on="Title", how="inner")

merged["merged_text"] = merged["summary"].fillna("") + ". " + merged["text"].fillna("")
merged = merged.drop_duplicates(subset="merged_text").reset_index(drop=True)

# compute average ratings
book_avg_ratings = merged.groupby("Title")["rating"].mean().reset_index()
book_avg_ratings.columns = ["Title", "avg_rating"]
merged = merged.merge(book_avg_ratings, on="Title", how="left")


[nltk_data] Downloading package wordnet to
[nltk_data]     /Users/rachelrieille/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     /Users/rachelrieille/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


The main approach used to extract general opinions from the reviews is the BERTopic function. It works in the following way :

#### 1. Sentence Embedding

Each document is transformed into a dense vector representation using a pretrained sentence embedding model. These models are based on transformer architectures (such as BERT) and are capable of capturing nuanced semantic relationships between texts.

#### 2. Dimensionality Reduction via UMAP

Given the high dimensionality of sentence embeddings, Uniform Manifold Approximation and Projection (UMAP) is applied to reduce these vectors to a lower-dimensional space.

#### 3. Clustering with HDBSCAN

The reduced embeddings are clustered using HDBSCAN, an algorithm that groups documents based on density. It does not necessary allocate each document to a cluster which avoids noise in the cluster.

#### 4. Topic Representation via TFIDF Vectorization

Once clusters (topics) are identified, the original documents associated with each cluster are analyzed using a vectorizer to extract the most representative terms. Custom stopword lists and text preprocessing steps (lemmatization) are applied to improve the specificity and interpretability of the topic representations.

We only use light preprocesing on the text, as the pretrained models are trained on raw data.

In [26]:
def minimal_clean(text):
    return re.sub(r'\s+', ' ', html.unescape(text)).strip()

merged['clean_text'] = merged['merged_text'].apply(minimal_clean)

To get topics related to good opinions and bad opinions, we will analyse and get topics from two separate datasets. The first will be the high ratings (4-5) and the second will be the bad ratings (1-2).

In [27]:
high_rated = merged[merged['rating'] >= 4][['clean_text', 'rating']]
low_rated = merged[merged['rating'] <= 2][['clean_text', 'rating']]

We will first check what are the most common word from the whole data set, to remove them later from our vocabulary (for the representation of the topics).

In [ ]:
# Basic vectorizer setup
vec = CountVectorizer(stop_words="english", ngram_range=(1, 1))
X = vec.fit_transform(merged['clean_text'])

# Sum word counts
word_counts = X.sum(axis=0).A1
words = vec.get_feature_names_out()

# Create a frequency DataFrame
freq_df = pd.DataFrame({"word": words, "count": word_counts})
freq_df = freq_df.sort_values(by="count", ascending=False)
print(freq_df.head(30))

We will remove contextual words that appear frequently, as well as qualtitative words that are not relevent to specific topics because we already divide our analysis by good or bad rating. We will also create our own tokenizer to input in BERTopic, to reduce the vocabulary used for the representation of each topics to its core.

In [28]:
from sklearn.feature_extraction import text

lemmatizer = WordNetLemmatizer()

customized_stopwords = {
    "book", "books", "read", "reading", "review", "reviews", "summary", 
    "story", "author", "novel", "characters", "character", "world",
    "just", "like", "really", "know", "way", "time", "people", "life", 
    "little", "did", "does", "don", "great", "amazing", "wonderful", "awesome", "excellent", "terrible", 
    "boring", "bad", "awful", "poor", "disappointing", "fantastic", "favorite", 
    "best", "worst", "loved", "hated"
}

def remove_stopwords_then_lemmatize(text):
    # Lowercase and remove punctuation
    text = text.lower()
    text = re.sub(r"[^a-z\s]", "", text)

    # Tokenize
    tokens = text.split()

    # Remove stopwords BEFORE lemmatizing
    tokens = [token for token in tokens if token not in combined_stopwords]

    # Lemmatize remaining tokens
    return [lemmatizer.lemmatize(token) for token in tokens]

combined_stopwords = list(text.ENGLISH_STOP_WORDS.union(customized_stopwords))

### What do people like ? 

In [2]:
top_books = (
    merged.groupby("Title")
    .agg(avg_rating=('avg_rating', 'mean'), review_count=('rating', 'count'))
    .query("review_count >= 100")
    .sort_values("avg_rating", ascending=False)
    .head(10)
)
display(top_books)


,avg_rating,review_count
Title,,
Harry Potter & the Prisoner of Azkaban,4.524715,263
Ella Enchanted,4.462185,119
The Hobbit,4.440233,343
Harry Potter and the Chamber of Secrets,4.436893,309
The Five Love Languages: The Secret to Love that Lasts,4.420561,107
"The Lord of the Rings Trilogy (The Fellowship of the Ring, The Two Towers, The Return of the King, I, II, III)",4.409836,183
"The Hobbit; Or, There and Back Again",4.403614,332
"The Hobbitt, or there and back again; illustrated by the author.",4.402332,343
Harry Potter and The Sorcerer's Stone,4.389855,690


These are the books with highest rating on average, focusing on books with at least 100 reviews. Readers seem to particularly like Fantasy books, such as the Hobbit or Harry Potter. 

In [8]:
umap_model = umap.UMAP(
    n_neighbors=15,       
    min_dist=0.3,         
    n_components=5,       
    metric="cosine",      
    random_state=SEED      # For reproducibility
)

hdbscan_model = HDBSCAN(
    min_cluster_size=100,       # Ensures each topic has significance
    min_samples=10,             # Slightly conservative clustering
    metric="euclidean",         # UMAP output is Euclidean
    prediction_data=True
)

vectorizer_model_high= TfidfVectorizer(
    stop_words=None,
    tokenizer=remove_stopwords_then_lemmatize, #use our custom tokenizer
    ngram_range=(1, 2),  
    min_df=40,
    max_df=0.8
)
sentence_model = SentenceTransformer("all-MiniLM-L6-v2")
representation_model = MaximalMarginalRelevance(diversity=0.6)

high_topic_model = BERTopic(representation_model=representation_model, 
                            hdbscan_model= hdbscan_model, 
                            embedding_model=sentence_model, 
                            vectorizer_model=vectorizer_model_high,
                            umap_model=umap_model)


high_topics, high_probs = high_topic_model.fit_transform(high_rated['clean_text'])

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The 

Once our topics are modelled, this type of object enables us to access directly visualization tools.

In [9]:
high_topic_model.visualize_barchart(top_n_topics=10)

To better interpet the keywords from the plot above, we can look at the representative docs. These are the documents that are most typical form that cluster (central).

In [10]:
high_topic_model.get_topic_info()[['Topic', 'Representative_Docs']].head(11)

,Topic,Representative_Docs
0,-1,"[Be careful of which edition you buy!. Fear and Trembling A Philosophical Masterpiece by Sren Kierkegaard is so short, you wonder why you would need notes and commmentary. You do! Get an edition w..."
1,0,"[Great collection of recipes. I wish this book were a little ""slicker"" with photographs for every recipe and heavy shiny paper, but I suppose that most publishers are reluctant to spend that kind ..."
2,1,"[To Hell and Back - One Hell of a Great Read. Sixty-one years ago, a young American who'd fought in the war published an unpretentious book, ""To Hell and Back."" It was the story of his experience ..."
3,2,"[Offers some mindblowing new perspectives on Christianity. What if Jesus wasn't divine? What if he was just a good man -- possibly a perfect one (whatever that means), but definitely mortal? This ..."
4,3,"[Great Purchase!!. The book arrived in perfect condition and on time. I would buy from this seller again!, Good Service!. I found the product to be in good shape and it was shipped per the seller'..."
5,4,[The Two Towers. This is the second book of the Lord of the Rings trilogy. It really kept me reading. I learned more about Frodo and Same and their quest to destroy the ring. While reading this bo...
6,5,[Good for beginners. This is an excellent book for people who knows nothing or too little about classical music. I read it and I found it funny and esay to read. It shows you why not to be afraid ...
7,6,[questions!. Children are full of question. This is a simple picture book that is full of questions. Such as where why do bunnies run? Where do birds fly and where do the whales sail. Young childr...
8,7,[George W. Bush is a good review.. I found this book to be encouraging and intriguing. I learned a lot about the heart and compassion of President Bush. The book will help the reader realize the d...
9,8,"[One of his best. i havent read this one for several years, after reading it again I still rate this as one of Stephen KIngs very best., Dreamcatcher. I am a HUGE Stephen King fan! And this book i..."


The "Topic Word Scores" graph for high-rated reviews shows what topics are the most frequently appearing in satisfied customers' reviews. Topic 0 with words like "recipe", “diet,” and “cook” suggests readers value content related to food and nutrition. Topic 1 centers on military history, especially World War II, indicating strong interest in well-written historical narratives.

Topic 2 covers Christian and spiritual books. Topic 3 reflects appreciation for the physical condition and delivery quality of books, showing the role of product satisfaction in positive ratings.

Topic 4 is likely linked to The Lord of the Rings, it seems to be a generally liked book. Topic 5 points to music-related content, suggesting readers enjoy this kind of content. Topic 6 pertains to children’s literature, especially bedtime stories and illustrated books.

Topic 7 features historical-political themes (e.g., "conservative", "lincoln"). Topic 8 showcases horror fiction, particularly Stephen King, highlighting high reader engagement with suspenseful narratives. Finally, Topic 9 combines thriller and romance genres.

### What do readers dislike ? 

In [ ]:
top_title = (
    merged.groupby("Title")
    .agg(avg_rating=('avg_rating', 'mean'), count=('rating', 'count'))
    .query("count >= 100")
    .sort_values("avg_rating", ascending=True)
    .head(10)
)
display(top_title)


Crossroads of Twilight (The Wheel of Time, Book 10), with a very bad rating average and a lot of reviews seems to be a particularly disliked book.

In [11]:
vectorizer_model_low = CountVectorizer(
    stop_words=None,
    tokenizer=remove_stopwords_then_lemmatize,
    ngram_range=(1, 2),   # Include unigrams
    min_df=40,
    max_df=0.8
)

representation_model = MaximalMarginalRelevance(diversity=0.5)

umap_model = umap.UMAP(
    n_neighbors=20,       # Slightly more global structure
    min_dist=0.3,         # Spacing between clusters
    n_components=5,       # Enough for separation but still compact
    metric="cosine",      # Ideal for embeddings like BERT
    random_state=42       # For reproducibility
)

hdbscan_model = HDBSCAN(
    min_cluster_size=50,       # Ensures each topic has significance
    min_samples=20,             # Slightly conservative clustering
    metric="euclidean",         # UMAP output is Euclidean
    prediction_data=True
)

low_topic_model = BERTopic(representation_model=representation_model, 
                           hdbscan_model= hdbscan_model, 
                           embedding_model=sentence_model, 
                           vectorizer_model=vectorizer_model_low,
                           umap_model=umap_model)

low_topics, low_probs = low_topic_model.fit_transform(low_rated['clean_text'])

In [13]:
low_topic_model.visualize_barchart(top_n_topics=10)

In [14]:
low_topic_model.get_topic_info()[['Topic', 'Representative_Docs']].head(11)

,Topic,Representative_Docs
0,-1,"[Genesis questioned by billions of years belief. While Hugh Ross insists on distinguishing himself from `theistic evolutionists', Ross adopts the same basic philosophical approach. That is, he mak..."
1,0,"[Oversexed, but at least it had a plot. This is an improvement over the past two books in the series, in that less than half the book is devoted to sex scenes and the other half actually deals wit..."
2,1,"[From normal to lucky. Jim Collins is trying to find the characteristics of what makes a good company a great company. His findings are supported by an extensive analysis of 1,435 companies for a ..."
3,2,[No page numbers. The description claims the Kindle edition has page numbers. I tried it on Kindle for iPad and Kindle for PC and guess what! There are no page numbers! I have the latest versions ...
4,3,"[Very sad. First, this whole ""conservative versus liberal"" game is old and dead. This whole paradigm only offers us 2 solutions to every problem, and 2 points of view. Have we become so unimaginat..."
5,4,[Disappointed. With the Wheel of Time series Robert Jordan created and introduced interesting characters involved in the familiar (but with a unique flare) theme of good versus evil. Crossroads of...
6,5,"[Some good dictionary innovations, but content a little thin. This dictionary, which is relatively small compared to other American dictionaries, has some excellent characteristics that I wish oth..."
7,6,"[No good ending. I liked it. But, then the ending messed it up. It has some vulgarity that wasn't really necessary. And, the end just didn't fit with the rest of the book., Needs a new ending. Thi..."
8,7,"[Another silly self-help book!. How can anyone advice a woman to give up her say in a marriage? Obviously this former shrew did. Cool, right? Not so. Think about it, is it smart to give up yoursel..."
9,8,"[Beautiful evocation of the non-historical Jesus. This is not about the historical Jesus, the author admits as much early on. And that is precisely the problem with this book. It is beautifully wr..."


In [15]:
low_topic_model.get_representative_docs(9)

['Item ordered 7/09/09 has not yet arrived. From Unsatisfied Buyer:I placed my order on Amazon July 9. Within one week I received the wrong version of book (v5), contacted seller, Best-Seller-Books, July 16 and requested v7. Seller replied "PLEASE EXCUSE US ABOUT THIS MISTAKE PLEASE KEEP THE WRONG BOOK AND I WILL CHECK MY INVENTORY IF I HAVE ANOTHER COPY OF THE 7TH EDITION OR GIVE YOU A FULL REFUND IF THAT IS GOOD FOR YOU CONTACT ME ASAP THANKS A LOT". My reply on July 16, "If you don\'t have 7th edition I\'ll need the refund. Please let me know if you will be sending 7th edition." Seller did not acknowledge this last reply from me to indicate if they had v7 in stock or not.On July 31 I contacted Best-Seller-Books (Seller) again expressing concern that I had not received an acknowledgement on the July 16 message. Was informed on same date that the seller had ordered this item and was waiting for it to arrive, and that this item would be shipped priority mail when the seller receives it

The topic distribution in low-rated reviews reveals several key drivers of dissatisfaction. Topic 0 highlights complaints about sexual or romantic content perceived as excessive (looking at the representative document). Topic 1 covers business and investment books, often seen as shallow or unhelpful. Topic 2 points to frustration with formatting and typographical errors, particularly in Kindle editions.

Topic 3 reflects disapproval of political content, as in the high rating reviews, but this time it might be due to the divergence of opinion between the author and the reader. Topic 4 refers to long fantasy series (e.g., Wheel of Time), with critiques about the plots. Topic 5 centers on language (dictionaries) and translation issues.

Topic 6 captures issues with predictable endings or bad plots. Topic 7 focuses on gender roles and relationship portrayals. Topic 8 relfects reviews related to religion. But looking at the representative documents, we get that the compalaints relate to theological disagreements and a sense of distortion of original Christian teachings. Finally, Topic 9 relates to negative late delivery experiences.



### Conclusion 



Looking at the reviews, we can see what people like and what they don’t. High ratings are often given to books that are fun, interesting, or useful, like cookbooks, history books, fantasy stories, and children’s books. People also like when the book arrives in good condition and matches what they expected.

On the other hand, low ratings are often given when books have problems like bad formatting, poor translation, or a weak story. Some people also don’t like books with strong opinions about politics or religion, especially when they don’t agree with them. Others dislike how some books talk about marriage or gender roles.

In short, readers enjoy books that are well-made, thoughtful, and match their interests. When books have technical issues or push ideas that readers strongly disagree with, they are more likely to get low ratings.

## Task 2.2

In [2]:

# Configurable variable for minimum number of reviewers
MIN_REVIEWERS = 1  # Adjust this to try different thresholds
MIN_RATING_AVERAGE = 3

# Load and preprocess data
books = pd.read_csv("ATML2025_books.csv")
train = pd.read_csv("ATML2025_book_reviews_train.csv")

# Print DataFrame size
print(f"Initial size of book list: {len(books)}")

# Allow user to specify custom size of Training set
# custom_size = int(input("Enter the number of books to process (or press Enter to use all): ") or books.shape[0])
# books = books.head(custom_size)
print(f"Processing {len(books)} books")

def preprocess(text):
    text = str(text)
    text = re.sub(r"http\S+|www\S+", "", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip().lower()

# Clean categories column (extract content between [' and '])
def clean_categories(category):
    if pd.isna(category) or category.strip() == "":
        return ""
    # Remove [' and '] (e.g., ['Law'] -> Law)
    if category.startswith("['") and category.endswith("']"):
        return category[2:-2].strip()
    return category.strip()

# Filter books with valid (non-null, non-empty) descriptions
books = books[books['description'].notnull() & (books['description'].str.strip() != "")].reset_index(drop=True)
print(f"Books with valid descriptions: {len(books)}")

# Preprocess description and categories
books["description"] = books['description'].apply(preprocess)
books["categories"] = books['categories'].apply(clean_categories)

# Repeat categories multiple times to give them more importance
books["combined_text"] = books["description"] + " " + (books["categories"] + " ") * 2

# Aggregate ratings and count reviewers using Title
ratings_summary = train.groupby('Title').agg({
    'rating': 'mean',
    'profileName': 'nunique'
}).rename(columns={'rating': 'avg_rating', 'profileName': 'num_reviewers'}).reset_index()
books = books.merge(ratings_summary, on='Title', how='left')
books['avg_rating'] = books['avg_rating'].fillna(books['avg_rating'].mean())
books['num_reviewers'] = books['num_reviewers'].fillna(0)

# Filter books with avg_rating > MIN_RATING_AVERAGE and num_reviewers >= MIN_REVIEWERS
books = books[(books['avg_rating'] > MIN_RATING_AVERAGE) & (books['num_reviewers'] >= MIN_REVIEWERS)].reset_index(drop=True)
print(f"Books with average rating > {MIN_RATING_AVERAGE} and at least {MIN_REVIEWERS} reviewers: {len(books)}")

# Check if there are enough books after filtering
if len(books) < 1:
    print("No books meet the criteria (valid description, rating > {}, reviewers >= {}). Try relaxing the filters.".format(MIN_RATING_AVERAGE,MIN_REVIEWERS))
    exit()

# Initialize sentence transformer model for semantic similarity
model = SentenceTransformer('all-MiniLM-L6-v2')  # Fast and effective for semantic search

# Generate embeddings for book descriptions
print('Encoding the descriptions...')
book_embeddings = model.encode(books["combined_text"].tolist(), convert_to_tensor=True, show_progress_bar=True)

# Recommendation function
def find_matching_books(query, top_n=5):
    # Preprocess query
    query = preprocess(query)
    
    # Generate query embedding
    query_embedding = model.encode([query], convert_to_tensor=True)
    
    # Compute cosine similarities
    similarities = cosine_similarity(query_embedding.cpu().numpy(), book_embeddings.cpu().numpy())[0]
    
    # Get top matching books
    top_indices = np.argsort(similarities)[::-1][:top_n]
    matching_books = books.iloc[top_indices][['Title', 'description', 'avg_rating', 'num_reviewers', 'categories']].copy()
    matching_books['similarity_score'] = similarities[top_indices]
    
    return matching_books.sort_values('similarity_score', ascending=False)

print('Code executed successfully!')

Initial size of book list: 108249
Processing 108249 books
Books with valid descriptions: 78594
Books with average rating > 3 and at least 1 reviewers: 39955
Encoding the descriptions...


Batches:   0%|          | 0/1249 [00:00<?, ?it/s]

Code executed successfully!


Here the function uses a pre-trained SentenceTransformer model to convert both the user's query and the book descriptions into vector embeddings (dense numerical representations of meaning).
These embeddings are then compared using cosine similarity to determine how close the user's intent is to the content of each book.
The books with the highest similarity scores are returned as recommendations.

###  Test 1

In [3]:
# Example usage
customer_query = "science fiction adventure with aliens"
recommended_books = find_matching_books(customer_query)
print("Recommended Books:")
print(recommended_books)

Recommended Books:
                                             Title  \
34309                             The Killing Star   
17357  Planet of Nightmares (Tom Swift Series #11)   
9376                             Pursuit (Roswell)   
37583   Vanderdeken's Children (Doctor Who Series)   
36867                              Superman/Aliens   

                                                                                                                                                                                                   description  \
34309  the opening chapter of an incredible adventure includes the destruction of earth by ten thousand relativistic bombs launched by an alien race in a science fiction thriller and follows the desperat...   
17357                                                        mr. swift takes several guests, tom, and his friends, to a mining planet owned by his company, but what begins as a vacation becomes a nightmare.   
9376   while liz, max, mar

The recommendations are well aligned with the query, as all selected titles fall within the science fiction genre and include themes such as space travel, alien encounters, and interstellar adventure. Books like The Killing Star, Planet of Nightmares, and Superman/Aliens clearly reflect the “aliens” and “adventure” aspect mentioned in the user’s input. The similarity scores, ranging from approximately 0.55 to 0.63, indicate a solid semantic match between the query and the book descriptions. Despite some titles having low numbers of reviews, their thematic relevance remains strong, which shows that the model is effectively prioritizing semantic content over popularity which is an important trait for content-based recommendations. Overall, the results confirm that the model accurately understands and retrieves books in line with the user's intent, especially for specific genre-based requests.

In [4]:
recommended_books['description'].values[0]

'the opening chapter of an incredible adventure includes the destruction of earth by ten thousand relativistic bombs launched by an alien race in a science fiction thriller and follows the desperate struggles of the remnants of humankind to survive in a hostile universe.'

### Test 2

In [5]:
customer_query = "history of second world war"
recommended_books = find_matching_books(customer_query)
print("Recommended Books:")
print(recommended_books)

Recommended Books:
                                                                                Title  \
31262                   Great Crusade: A New Complete History of the Second World War   
5968   The Soldier and the State: The Theory and Politics of Civil-Military Relations   
4459                                   A war to be won: Fighting the Second World War   
28261                        No End Save Victory Vol. 1: Perspectives on World War II   
3146                                                  History of the Second World War   

                                                                                                                                                                                                   description  \
31262                                                                                                                              an updated edition of the classic survey of world war ii's military history   
5968   world war ii: the a

The model continues to perform well, accurately identifying books focused on World War II history in response to the query. Titles such as Great Crusade and A War to Be Won directly address the Second World War, and all recommended books are categorized under history. The high similarity scores (up to 0.72) confirm a strong semantic match, reinforcing the model’s ability to interpret factual, topic-specific queries effectively.

In [6]:
recommended_books['description'].values[1]

'world war ii: the alchemy of power; civil-military relations in the postwar decade; the political roles of the joints chiefs; the separation of power and the cold war defense; departmental structure of civil-military relations; toward a new equilibrium.'

### Test 3 - rare and specific keywords

In [7]:
customer_query = "cigars"
recommended_books = find_matching_books(customer_query)
print("Recommended Books:")
print(recommended_books)

Recommended Books:
                                                                Title  \
1431                                      Cubans: The Ultimate Cigars   
2050                                                 The Cigar in Art   
37705  Trading With The Enemy: A Yankee Travels Through Castro's Cuba   
33022            A Stranger In The Barrio: Memoir of a Tampa Sicilian   
7594                Collectible Ashtrays: Information and Price Guide   

                                                                                                                                                                                                   description  \
1431   the most definitive guide to cuban cigars: the cuban cigar handbook profiles the history of cigars in cuba and features an extensive guide to over 200 varieties. for more than two centuries, cuban...   
2050   the ultimate art book for the cigar lover, the cigar in art is a celebration of cigars and the good things in life.

For this niche query, the model still performs reasonably well by retrieving books directly related to cigars, such as Cubans: The Ultimate Cigars and The Cigar in Art. While the relevance slightly drops with the lower-ranked results, the top recommendations are thematically accurate, showing that the model can handle even very specific or uncommon topics to a useful degree.

In [8]:
recommended_books['description'].values[0]

"the most definitive guide to cuban cigars: the cuban cigar handbook profiles the history of cigars in cuba and features an extensive guide to over 200 varieties. for more than two centuries, cuban cigars have been heralded as the best cigars in the world. more than just a cigar, they're an art form, with tobacco growers and hand-rollers considered artists. today, there are more than 200 varieties to discover, and this essential guide highlights each one. featuring insights from industry experts like gary korb and denis k. toulouse, the cuban cigar handbook presents an in-depth look at a wide range of fascinating topics, including: - a complete history of cuban cigars - how to spot fakes - stories of celebrated cigar aficionados from ernest hemingway to rudyard kipling - the best cuban rum to pair with a cigar - vivid descriptions of cuba and its environs - dynamic profiles of growers, hand-rollers, and producers - and so much more! the cuban cigar handbook tells the history of cigars 

## Recommendation function in practice

We implemented a UI to provide recommendations in a more user-friendly way.

In [9]:
def launch_recommender_ui():
    # Create widgets
    query_box = widgets.Text(
        value='',
        placeholder='Describe your interest (e.g. vegan food, fantasy novels, tech startups)...',
        description='What kind of book are you looking for ?',
        style={'description_width': 'initial'},
        layout=widgets.Layout(width='80%')
    )

    top_n_slider = widgets.IntSlider(
        value=5,
        min=1,
        max=10,
        step=1,
        description='Number of results:',
        style={'description_width': 'initial'},
        layout=widgets.Layout(width='40%')
    )

    search_button = widgets.Button(
        description='Look for recommendations',
        button_style='primary',
        tooltip='Click to find recommended books',
        layout=widgets.Layout(width='25%')
    )

    output = widgets.Output()

    def on_search_button_clicked(b):
        output.clear_output()
        with output:
            query = query_box.value.strip()
            if not query:
                display(Markdown("\n You need to enter a query to get book recommendations !"))
                return
            results = find_matching_books(query, top_n=top_n_slider.value)
            display(Markdown(f"### Top {len(results)} Recommendations for: *{query}*"))
            for _, row in results.iterrows():
                display(Markdown(
                    f"---\n"
                    f"### **{row['Title']}**\n"
                    f"**Rating:** {row['avg_rating']:.2f} ({int(row['num_reviewers'])} reviewers)  \n"
                    f"**Category:** *{row['categories']}*  \n"
                    f"**Description:** {row['description'][:300]}{'...' if len(row['description']) > 300 else ''}  \n"
                    f"**Similarity Score:** `{row['similarity_score']:.4f}`"
                ))

    # Bind handler safely
    search_button.on_click(on_search_button_clicked)

    # Display the UI
    ui = widgets.VBox([query_box, top_n_slider, search_button, output])
    display(ui)


In [10]:
launch_recommender_ui()

## Limitations of the recomendation system 

1. "all-MiniLM-L6-v2" is a very light model in this category. It has a max sequence length of 256 and embeds the phrases in 384 dimensions. This may in some cases be insufficient if the texts in input are very large. We could try to improve this by using a more complex model with more parameters, like "all-mpnet-base-v2", which is 5 times larger, but then we will have a hardware limitation, it could become too difficult to compute on a simple laptop.

2. The preprocess function removes URLs and extra spaces but does not handle punctuation, special characters, or stop words, which could affect the quality of embeddings and similarity matching. A gridsearch could help find the best parameters but for that we would need to have a large labeled dataset which we don't have. 

3. The size of the data could be a limitation. The current size of data was processed and embedded in a timely manner on a normal computer but it can very quickly become slow because of lack of RAM or processing units
   
4.  Here the book recommendation is based in the cosine similarity of the two semtence embeddings (the query and the description of the book) and it works well enough in simple cases. But if the input query is too complex and nuanced with multiple intents this may decrease its performance.

###  Declaration of Generative AI Tool Usage

ChatGPT (OpenAI, GPT-4) was used as a generative AI tool to support the development of this assignment.

ChatGPT was asked for assistance in order to draft functions and debugging.

All implementation, fine-tuning, and interpretation were conducted by our team. We wrote the final code, results analysis, and structured the notebook.